# SNOMED CT Entity Linking with SynthLab

This notebook demonstrates how to use SynthLab's SNOMED CT entity linking capabilities to ground medical entities to standardized clinical terminology.

**Features covered:**
- Basic entity linking with `SNOMEDLinker`
- Multiple embedding model options (SapBERT, Qwen3, etc.)
- Grounding causal graphs from SOAP notes
- Two-stage pipeline: LLM extraction + embedding-based linking

## Why SNOMED CT?

SNOMED CT (Systematized Nomenclature of Medicine - Clinical Terms) is the most comprehensive clinical terminology system, containing over 350,000 concepts. Grounding free-text medical entities to SNOMED CT provides:

- **Standardization**: Same concept ID regardless of how it's written ("heart attack" = "MI" = "myocardial infarction")
- **Interoperability**: Enables integration with EHR systems, research databases, and clinical decision support
- **Hierarchical relationships**: SNOMED CT includes IS-A relationships for reasoning

## Setup

In [1]:
import synthlab as sl

# Check SNOMED linking status
sl.print_snomed_info()

/home/schilder/.conda/envs/synthlab/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[01/25/26 11:20:43] INFO     Loading faiss.                                                           ]8;id=286150;file:///home/schilder/.conda/envs/synthlab/lib/python3.12/site-packages/faiss/loader.py\loader.py]8;;\:]8;id=983022;file:///home/schilder/.conda/envs/synthlab/lib/python3.12/site-packages/faiss/loader.py#133\133]8;;\

                    INFO     Successfully loaded faiss.                                               ]8;id=979511;file:///home/schilder/.conda/envs/synthlab/lib/python3.12/site-packages/faiss/loader.py\loader.py]8;;\:]8;id=995583;file:///home/schilder/.conda/envs/synthlab/lib/python3.12/site-packages/faiss/loader.py#135\135]8;;\


  ░██████╗██╗░░░██╗███╗░░██╗████████╗██╗░░██╗  ██╗░░░░░░█████╗░██████╗░
  ██╔════╝╚██╗░██╔╝████╗░██║╚══██╔══╝██║░░██║  ██║░░░░░██╔══██╗██╔══██╗
  ╚█████╗░░╚████╔╝░██╔██╗██║░░░██║░░░███████║  ██║░░░░░███████║██████╦╝
  ░╚═══██╗░░╚██╔╝░░██║╚████║░░░██║░░░██╔══██║  ██║░░░░░██╔══██║██╔══██╗
  ██████╔╝░░░██║░░░██║░╚███║░░░██║░░░██║░░██║  ███████╗██║░░██║██████╦╝
  ╚═════╝░░░░╚═╝░░░╚═╝░░╚══╝░░░╚═╝░░░╚═╝░░╚═╝  ╚══════╝╚═╝░░╚═╝╚═════╝░
        ▌║█║▌│║▌│║▌║▌█║▌║█║▌│║▌│║▌║▌█║▌║█║▌│║▌│║▌║▌█║▌│║▌│║▌║▌█║
  ════════════════════════════════════════════════════════════════════
  Synthetic Healthcare Data Toolkit
  ────────────────────────────────────────────────────────────────────
  ◈ EHR        Synthetic patient records (diagnoses, meds, labs)
  ◈ Genomics   Synthetic genotypes with realistic LD structure
  ◈ Imaging    Datasets + synthetic generation (CT, MRI, X-ray)
  ◈ Multimodal Linked EHR + Imaging + Genomics per patient
  ◈ AI Notes   SOAP notes with causal graph analysis
  ══════════════════

## 1. Basic Entity Linking with SNOMEDLinker

The `SNOMEDLinker` uses semantic embeddings to find the closest SNOMED CT concepts for a given medical term.

In [2]:
# Set up a linker with sample SNOMED concepts (no license required)
# This builds a FAISS index from ~200 common clinical concepts
linker = sl.setup_sample_linker()

Building sample SNOMED index (first time only)...
Loading embedding model: cambridgeltl/SapBERT-from-PubMedBERT-fulltext
  Device: cuda
Building SNOMED index from 167 concepts
  Encoding terms with cambridgeltl/SapBERT-from-PubMedBERT-fulltext...


Encoding: 100%|██████████| 2/2 [00:00<00:00, 13.25batch/s]

  Embeddings shape: (167, 768)
  Building FAISS index...
  Index contains 167 vectors
  Saving index to /home/schilder/.cache/synthlab/snomed/snomed_sapbert.index
  Saved concepts to /home/schilder/.cache/synthlab/snomed/snomed_concepts.json


In [3]:
# Link a single term
matches = linker.link("heart attack")

print("Top matches for 'heart attack':")
for match in matches[:3]:
    print(f"  {match}")

Top matches for 'heart attack':
  SCTID:22298006 | Myocardial infarction (score: 0.778)
  SCTID:194828000 | Angina pectoris (score: 0.727)
  SCTID:230690007 | Cerebrovascular accident (score: 0.669)


In [4]:
# Link multiple terms at once (more efficient)
terms = ["diabetes", "high blood pressure", "stroke", "kidney failure"]
results = linker.link_batch(terms, k=2)

print("Batch linking results:")
for term, matches in results:
    top_match = matches[0] if matches else None
    if top_match:
        print(f"  '{term}' -> SCTID:{top_match.concept_id} | {top_match.term} (score: {top_match.score:.3f})")
    else:
        print(f"  '{term}' -> No match")

Batch linking results:
  'diabetes' -> SCTID:73211009 | Diabetes mellitus (score: 0.972)
  'high blood pressure' -> SCTID:38341003 | Hypertension (score: 0.895)
  'stroke' -> SCTID:230690007 | Cerebrovascular accident (score: 0.812)
  'kidney failure' -> SCTID:236423003 | Renal impairment (score: 0.831)


## 2. Available Embedding Models

SynthLab supports multiple embedding models for entity linking. Each has different tradeoffs:

In [5]:
# View available models
print("Available embedding models:")
print()
for name, model_id in sl.EMBEDDING_MODELS.items():
    print(f"  {name:15} -> {model_id}")

Available embedding models:

  sapbert         -> cambridgeltl/SapBERT-from-PubMedBERT-fulltext
  sapbert-mean    -> cambridgeltl/SapBERT-from-PubMedBERT-fulltext-mean-token
  sapbert-xlm     -> cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR
  qwen3-0.6b      -> Qwen/Qwen3-Embedding-0.6B
  gte-qwen2       -> Alibaba-NLP/gte-Qwen2-1.5B-instruct
  gte-large       -> thenlper/gte-large
  all-mpnet       -> sentence-transformers/all-mpnet-base-v2
  bge-large       -> BAAI/bge-large-en-v1.5


### Model Comparison

| Model | Size | Best For | Notes |
|-------|------|----------|-------|
| `sapbert` | 110M | Biomedical entities | Trained on UMLS, best accuracy for clinical terms |
| `qwen3-0.6b` | 600M | General + speed | Efficient, good quality, instruction-tuned |
| `gte-qwen2` | 1.5B | High quality | State-of-the-art general embeddings |
| `bge-large` | 335M | General purpose | Good balance of speed and quality |

In [12]:
# Create a linker with a different model
# Note: This will download the model on first use

# Example: Use Qwen3 embeddings (uncomment to try)
linker_qwen = sl.SNOMEDLinker(
    model_id=sl.EMBEDDING_MODELS["qwen3-0.6b"],
    verbose=True
)

# # Build index with sample concepts
concepts = sl.get_sample_snomed_concepts()
linker_qwen.build_index(concepts)

Loading embedding model: Qwen/Qwen3-Embedding-0.6B
  sentence-transformers not available, falling back to transformers
  Device: cuda
Building SNOMED index from 167 concepts
  Encoding terms with Qwen/Qwen3-Embedding-0.6B...


Encoding: 100%|██████████| 2/2 [00:00<00:00,  5.24batch/s]

  Embeddings shape: (167, 1024)
  Building FAISS index...
  Index contains 167 vectors
  Saving index to /home/schilder/.cache/synthlab/snomed/snomed_sapbert.index
  Saved concepts to /home/schilder/.cache/synthlab/snomed/snomed_concepts.json


## 3. Loading SNOMED Concepts

SynthLab provides several ways to load SNOMED concepts:

In [13]:
# Option 1: Sample concepts (no license required, ~200 common terms)
sample_concepts = sl.get_sample_snomed_concepts()
print(f"Sample concepts: {len(sample_concepts)} terms")
print("\nExample concepts:")
for c in sample_concepts[:5]:
    print(f"  {c}")

Sample concepts: 167 terms

Example concepts:
  SCTID:22298006 (Myocardial infarction)
  SCTID:53741008 (Coronary artery disease)
  SCTID:38341003 (Hypertensive disorder)
  SCTID:49436004 (Atrial fibrillation)
  SCTID:84114007 (Heart failure)


In [14]:
# Option 2: Load from CSV (your own data)
# concepts = sl.load_snomed_from_csv(
#     "path/to/snomed.csv",
#     concept_id_col="sctid",
#     term_col="term"
# )

# Option 3: Load from UMLS (requires UMLS license)
# concepts = sl.load_snomed_from_umls("/path/to/UMLS/META")

## 4. Grounding Causal Graphs from SOAP Notes

The most powerful use case is grounding causal graphs extracted from clinical notes. This converts free-text medical concepts to standardized SNOMED CT IDs.

In [15]:
# Example: Parse a causal graph from text
causal_text = """
Obesity[lifestyle] ++> Diabetes[condition]
Diabetes[condition] ++> Diabetic_Nephropathy[condition]
Diabetes[condition] ++> Diabetic_Retinopathy[condition]
Hypertension[condition] ++> Chronic_Kidney_Disease[condition]
Metformin[medication] --> Blood_glucose[finding]
Smoking[lifestyle] ++> Lung_cancer[condition]
"""

graph = sl.parse_causal_graph(causal_text)
print(f"Parsed {len(graph.edges)} causal relationships")
print(f"Unique nodes: {len(graph.get_nodes())}")

Parsed 6 causal relationships
Unique nodes: 10


In [16]:
# Ground the causal graph to SNOMED CT
grounded = graph.ground_to_snomed(linker=linker, threshold=0.3)

print("\nGrounded nodes:")
for node in grounded.nodes:
    print(f"  {node.mention:25} -> SCTID:{node.concept_id:12} | {node.term} (confidence: {node.confidence:.2f})")

Grounding 10 nodes to SNOMED CT...
  Grounded: 10/10 nodes, 6/6 edges

Grounded nodes:
  Lung_cancer[condition]    -> SCTID:254637007    | Lung cancer (confidence: 0.96)
  Blood_glucose[finding]    -> SCTID:165816005    | Elevated blood glucose (confidence: 0.69)
  Metformin[medication]     -> SCTID:387458008    | Metformin (confidence: 0.99)
  Diabetic_Retinopathy[condition] -> SCTID:709044004    | Diabetic nephropathy (confidence: 0.69)
  Smoking[lifestyle]        -> SCTID:77176002     | Smoker (confidence: 0.80)
  Chronic_Kidney_Disease[condition] -> SCTID:431855005    | Chronic kidney disease (confidence: 0.96)
  Obesity[lifestyle]        -> SCTID:414916001    | Obesity (confidence: 0.95)
  Hypertension[condition]   -> SCTID:38341003     | Hypertension (confidence: 0.99)
  Diabetes[condition]       -> SCTID:73211009     | Diabetes mellitus (confidence: 0.97)
  Diabetic_Nephropathy[condition] -> SCTID:709044004    | Diabetic nephropathy (confidence: 0.98)


In [10]:
# View grounded edges
print("\nGrounded causal relationships:")
for edge in grounded.edges:
    print(f"  {edge.source.term} {edge.relation} {edge.target.term}")
    print(f"    (SCTID:{edge.source.concept_id} -> SCTID:{edge.target.concept_id})")


Grounded causal relationships:
  Obesity ++> Diabetes mellitus
    (SCTID:414916001 -> SCTID:73211009)
  Diabetes mellitus ++> Diabetic nephropathy
    (SCTID:73211009 -> SCTID:709044004)
  Diabetes mellitus ++> Diabetic nephropathy
    (SCTID:73211009 -> SCTID:709044004)
  Hypertension ++> Chronic kidney disease
    (SCTID:38341003 -> SCTID:431855005)
  Metformin --> Elevated blood glucose
    (SCTID:387458008 -> SCTID:165816005)
  Smoker ++> Lung cancer
    (SCTID:77176002 -> SCTID:254637007)


In [11]:
# Export to dictionary for downstream use
grounded_dict = grounded.to_dict()
print("Grounded graph as dict:")
print(f"  Nodes: {len(grounded_dict['nodes'])}")
print(f"  Edges: {len(grounded_dict['edges'])}")

Grounded graph as dict:
  Nodes: 10
  Edges: 6


## 5. Integration with SOAPNoteGenerator

The easiest way to use SNOMED grounding is directly through `SOAPNoteGenerator`:

In [ ]:
# Option 1: Enable grounding at initialization
# generator = sl.SOAPNoteGenerator(
#     ground_snomed=True,
#     snomed_embedding_model="sapbert"  # or "qwen3-0.6b" for efficiency
# )
# 
# # Generate SOAP note - causal graph is automatically grounded
# soap_note = generator.generate(patient)
# 
# # Access grounded graph
# for node in soap_note.grounded_causal_graph.nodes:
#     print(f"{node.mention} -> SCTID:{node.concept_id}")

In [ ]:
# Option 2: Enable grounding per-call
# generator = sl.SOAPNoteGenerator()
# 
# # Only ground this specific note
# soap_note = generator.generate(
#     patient,
#     ground_snomed=True,
#     snomed_embedding_model="qwen3-0.6b"
# )

In [ ]:
# Option 3: Ground after generation (most flexible)
# soap_note = generator.generate(patient)
# 
# # Ground the causal graph manually
# grounded = soap_note.extract_grounded_causal_graph(
#     embedding_model="sapbert",
#     threshold=0.5
# )

## 6. Two-Stage Pipeline: LLM Extraction + Embedding Linking

For best accuracy, use a two-stage pipeline:
1. **LLM extraction**: Use an LLM to extract and standardize medical entities
2. **Embedding linking**: Link standardized terms to SNOMED CT

This handles abbreviations, synonyms, and local terminology variations.

In [ ]:
# Create the two-stage pipeline
# Note: Requires an LLM API (Gemini, OpenAI, etc.)

# extractor, linker = sl.create_entity_pipeline(
#     embedding_model="cambridgeltl/SapBERT-from-PubMedBERT-fulltext",
#     llm_model="gemini-2.0-flash",  # or "gpt-4", "claude-3", etc.
#     use_llm_extraction=True
# )

In [ ]:
# Example: Extract and link entities from clinical text
# clinical_text = """
# Patient presents with worsening SOB and bilateral LE edema.
# History of CHF, DM2, and CKD stage 3. Currently on lasix 40mg,
# metformin 1000mg BID, and lisinopril 10mg daily.
# """
# 
# # Stage 1: LLM extracts and standardizes entities
# # Stage 2: Standardized terms are linked to SNOMED CT
# linked_entities = extractor.extract_and_link(clinical_text, linker)
# 
# for entity in linked_entities:
#     print(f"{entity.mention:20} -> {entity.top_match}")

## 7. Understanding Match Scores

The similarity score indicates how confident the linker is in the match:

| Score Range | Interpretation |
|-------------|----------------|
| 0.9 - 1.0 | Exact or near-exact match |
| 0.7 - 0.9 | High confidence match |
| 0.5 - 0.7 | Moderate confidence, review recommended |
| < 0.5 | Low confidence, likely incorrect |

In [ ]:
# Example: Different confidence levels
test_terms = [
    "myocardial infarction",  # Exact SNOMED term
    "heart attack",           # Common synonym
    "MI",                     # Abbreviation
    "cardiac event",          # Vague term
]

results = linker.link_batch(test_terms, k=1)

print("Confidence comparison:")
for term, matches in results:
    if matches:
        m = matches[0]
        confidence = "HIGH" if m.score > 0.7 else "MEDIUM" if m.score > 0.5 else "LOW"
        print(f"  '{term}' -> {m.term} (score: {m.score:.3f}, {confidence})")

## 8. Exporting Grounded Data

Grounded graphs can be exported for downstream analysis:

In [ ]:
# Export to dictionary
data = grounded.to_dict()

# Convert to JSON
import json
json_str = json.dumps(data, indent=2)
print("JSON export (first 500 chars):")
print(json_str[:500])

In [ ]:
# Export to NetworkX for graph analysis
try:
    G = grounded.to_networkx()
    print(f"NetworkX graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
    
    # Example: Find all nodes that influence kidney disease
    import networkx as nx
    kidney_nodes = [n for n, d in G.nodes(data=True) if 'kidney' in d.get('term', '').lower()]
    for node in kidney_nodes:
        predecessors = list(G.predecessors(node))
        print(f"  Factors influencing {G.nodes[node]['term']}: {predecessors}")
except ImportError:
    print("NetworkX not installed. Install with: pip install networkx")

## Summary

SynthLab provides a complete pipeline for SNOMED CT entity linking:

1. **`SNOMEDLinker`**: Core class for embedding-based entity linking
2. **Multiple embedding models**: SapBERT (biomedical), Qwen3 (efficient), and more
3. **Causal graph grounding**: `graph.ground_to_snomed()` or `SOAPNote.extract_grounded_causal_graph()`
4. **SOAPNoteGenerator integration**: `ground_snomed=True` for automatic grounding
5. **Two-stage pipeline**: `MedicalEntityExtractor` + `SNOMEDLinker` for best accuracy

### Key Functions

```python
# Quick setup
linker = sl.setup_sample_linker()

# Link entities
matches = linker.link("heart attack")
results = linker.link_batch(["diabetes", "hypertension"])

# Ground causal graphs
grounded = graph.ground_to_snomed(linker)

# With SOAPNoteGenerator
generator = sl.SOAPNoteGenerator(ground_snomed=True)
```